In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.globalfirepower.com/aircraft-total-fighters.php"

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
except requests.RequestException as e:
    print("Error fetching the page:", e)
    exit()

soup = BeautifulSoup(response.text, "lxml")

countries = soup.find_all("a", href=True)

data = []

for country in countries:
    # Rank
    rank_tag = country.select_one(".rankNumContainer span")
    rank = rank_tag.text.strip() if rank_tag else None

    # Full Country Name
    full_name_tag = country.select_one(".longFormName span")
    full_name = full_name_tag.text.strip() if full_name_tag else None

    # Short Country Name
    short_name_tag = country.select_one(".shortFormName span")
    short_name = short_name_tag.text.strip() if short_name_tag else None

    # Value
    value_tag = country.select_one(".valueContainer span span")
    value = value_tag.text.strip() if value_tag else None

    # Clean value (remove commas, convert to int)
    if value:
        value = value.replace(",", "")
        value = int(value) if value.isdigit() else None

    # Store only valid rows
    if rank and full_name and value is not None:
        data.append({
            "Rank": int(rank),
            "Country_Full_Name": full_name,
            "Country_Short_Name": short_name,
            "Value": value
        })

# Convert to DataFrame
df = pd.DataFrame(data)

# Save files
df.to_csv("/content/drive/MyDrive/Colab Notebooks/project/fighter_aircraft_data.csv", index=False)
df.to_excel("/content/drive/MyDrive/Colab Notebooks/project/fighter_aircraft_data.xlsx", index=False)

print(url, "Scraping is Completed.")
print("Files saved:")
print("- fighter_aircraft_data.csv")
print("- fighter_aircraft_data.xlsx")


https://www.globalfirepower.com/aircraft-total-fighters.php Scraping is Completed.
Files saved:
- fighter_aircraft_data.csv
- fighter_aircraft_data.xlsx


In [ ]:
for d in data[:5]:   # show first 5
    print(d)

{'Rank': 1, 'Country_Full_Name': 'United States', 'Country_Short_Name': 'USA', 'Value': 1790}
{'Rank': 2, 'Country_Full_Name': 'China', 'Country_Short_Name': 'CHN', 'Value': 1212}
{'Rank': 3, 'Country_Full_Name': 'Russia', 'Country_Short_Name': 'RUS', 'Value': 833}
{'Rank': 4, 'Country_Full_Name': 'India', 'Country_Short_Name': 'IND', 'Value': 513}
{'Rank': 5, 'Country_Full_Name': 'North Korea', 'Country_Short_Name': 'NKO', 'Value': 368}


In [ ]:
df = pd.DataFrame(data)
df

,Rank,Country_Full_Name,Country_Short_Name,Value
0,1,United States,USA,1790
1,2,China,CHN,1212
2,3,Russia,RUS,833
3,4,India,IND,513
4,5,North Korea,NKO,368
...,...,...,...,...
140,141,South Sudan,SSD,0
141,142,Suriname,SRN,0
142,143,Tajikistan,TJK,0
143,144,Uruguay,URU,0


In [ ]:
df.tail()

,Rank,Country_Full_Name,Country_Short_Name,Value
140,141,South Sudan,SSD,0
141,142,Suriname,SRN,0
142,143,Tajikistan,TJK,0
143,144,Uruguay,URU,0
144,145,Zambia,ZAM,0


In [ ]:
df.head()

,Rank,Country_Full_Name,Country_Short_Name,Value
0,1,United States,USA,1790
1,2,China,CHN,1212
2,3,Russia,RUS,833
3,4,India,IND,513
4,5,North Korea,NKO,368


In [ ]:
df.to_csv("/content/drive/MyDrive/Colab Notebooks/project/links_for_military_data.txt", index=False)
#df.to_csv(r"C:\Users\Rayala Kusumanjali\OneDrive\Documents", index=False)
print("CSV file saved successfully ")

CSV file saved successfully 


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://www.globalfirepower.com/countries-listing.php"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

print("Scraping Power Index from base URL...")

# -------------------------
# Request with error handling
# -------------------------
try:
    response = requests.get(BASE_URL, headers=HEADERS, timeout=10) # downloads the webpage
    response.raise_for_status() # if sometings goes wrong, then the program stops safely
except requests.RequestException as e:
    print("Failed to fetch page:", e)
    exit()

soup = BeautifulSoup(response.text, "lxml")

rows = []

cards = soup.select("a[href*='country-military-strength-detail']")

for card in cards:
    rank_tag = card.select_one(".rankNumContainer span")
    full_name_tag = card.select_one(".longFormName span")
    short_name_tag = card.select_one(".shortFormName span")
    pwr_tag = card.select_one(".pwrIndxContainer span")

    if rank_tag and full_name_tag and pwr_tag:
        # Clean values
        rank = rank_tag.text.strip()
        full_name = full_name_tag.text.strip()
        short_name = short_name_tag.text.strip() if short_name_tag else None

        power_index = (
            pwr_tag.text
            .replace("PwrIndx:", "")
            .replace("PwrIndx", "")
            .strip()
        )

        rows.append({
            "Rank": int(rank) if rank.isdigit() else rank,
            "Country_Full_Name": full_name,
            "Country_Short_Name": short_name,
            "Power_Index": power_index
        })


Scraping Power Index from base URL...


In [ ]:
df_power_index = pd.DataFrame(rows)

OUTPUT_PATH ="/content/drive/MyDrive/Colab Notebooks/project/power_index_countries.csv"
df_power_index.to_csv(OUTPUT_PATH, index=False)
print("Power Index scraping completed")

print(f"File saved: {OUTPUT_PATH}")
print(f"Total countries scraped: {df_power_index.shape[0]}")


Power Index scraping completed
File saved: /content/drive/MyDrive/Colab Notebooks/project/power_index_countries.csv
Total countries scraped: 145


In [ ]:
df_power_index.head() # it converts country data into pandas dataframe

,Rank,Country_Full_Name,Country_Short_Name,Power_Index
0,1,United States,USA,0.0744
1,2,Russia,RUS,0.0788
2,3,China,CHN,0.0788
3,4,India,IND,0.1184
4,5,South Korea,SKO,0.1656


In [ ]:
import re

# Read the config file
with open("/content/drive/MyDrive/Colab Notebooks/project/links_for_military_data.txt", "r", encoding="utf-8") as file:
    content = file.read()
print(content)
# Regex to extract URLs inside other_sources
urls = re.findall(
    r"'(https://www\.globalfirepower\.com/[^']+\.php)'",
    content
)

# Remove base_url if present
urls = list(set(urls))
urls.sort()

print(f"Total URLs extracted: {len(urls)}")
urls

Rank,Country_Full_Name,Country_Short_Name,Value
1,United States,USA,1790
2,China,CHN,1212
3,Russia,RUS,833
4,India,IND,513
5,North Korea,NKO,368
6,Pakistan,PAK,328
7,South Korea,SKO,315
8,Taiwan,TWN,285
9,Saudi Arabia,SAR,283
10,Israel,ISR,240
11,Egypt,EGY,238
12,France,FRA,226
13,Japan,JPN,217
14,Turkiye,TKY,201
15,Iran,IRN,188
16,Greece,GRE,178
17,Spain,SPN,137
18,Germany,GER,129
19,United Kingdom,UKD,113
20,Syria,SYR,104
21,Algeria,ALG,102
22,Qatar,QTR,102
23,Singapore,SNG,100
24,United Arab Emirates,UAE,99
25,Italy,ITA,89
26,Morocco,MOR,83
27,Thailand,THL,72
28,Angola,ANG,71
29,Sweden,SWE,71
30,Ukraine,UKR,70
31,Canada,CAN,66
32,Kazakhstan,KAZ,63
33,Poland,POL,59
34,Myanmar,MYA,58
35,Uzbekistan,UZB,58
36,Finland,FIN,54
37,Chile,CHI,45
38,Jordan,JOR,44
39,Belgium,BEL,43
40,Brazil,BRA,43
41,Kuwait,KUW,43
42,Switzerland,SWZ,43
43,Bangladesh,BNG,42
44,Indonesia,INO,41
45,Vietnam,VET,41
46,Sudan,SDN,37
47,Belarus,BLR,36
48,Netherlands,NTH,32
49,Denmark,DEN,31
50,Venezuela,VEN,30
51,Oman

[]

In [ ]:
urls = [
    "https://www.globalfirepower.com/total-population-by-country.php",
    "https://www.globalfirepower.com/available-military-manpower.php",
    "https://www.globalfirepower.com/manpower-fit-for-military-service.php",
    "https://www.globalfirepower.com/manpower-reaching-military-age-annually.php",
    "https://www.globalfirepower.com/active-military-manpower.php",
    "https://www.globalfirepower.com/active-reserve-military-manpower.php",
    "https://www.globalfirepower.com/manpower-paramilitary.php",

    "https://www.globalfirepower.com/aircraft-total.php",
    "https://www.globalfirepower.com/aircraft-total-fighters.php",
    "https://www.globalfirepower.com/aircraft-total-attack-types.php",
    "https://www.globalfirepower.com/aircraft-total-transports.php",
    "https://www.globalfirepower.com/aircraft-total-trainers.php",
    "https://www.globalfirepower.com/aircraft-total-special-mission.php",
    "https://www.globalfirepower.com/aircraft-total-tanker-fleet.php",
    "https://www.globalfirepower.com/aircraft-helicopters-total.php",
    "https://www.globalfirepower.com/aircraft-helicopters-attack.php",

    "https://www.globalfirepower.com/armor-tanks-total.php",
    "https://www.globalfirepower.com/armor-apc-total.php",
    "https://www.globalfirepower.com/armor-self-propelled-guns-total.php",
    "https://www.globalfirepower.com/armor-towed-artillery-total.php",
    "https://www.globalfirepower.com/armor-mlrs-total.php",

    "https://www.globalfirepower.com/navy-ships.php",
    "https://www.globalfirepower.com/navy-force-by-tonnage.php",
    "https://www.globalfirepower.com/navy-aircraft-carriers.php",
    "https://www.globalfirepower.com/navy-helo-carriers.php",
    "https://www.globalfirepower.com/navy-submarines.php",
    "https://www.globalfirepower.com/navy-destroyers.php",
    "https://www.globalfirepower.com/navy-frigates.php",
    "https://www.globalfirepower.com/navy-corvettes.php",
    "https://www.globalfirepower.com/navy-patrol-coastal-craft.php",
    "https://www.globalfirepower.com/navy-mine-warfare-craft.php",

    "https://www.globalfirepower.com/defense-spending-budget.php",
    "https://www.globalfirepower.com/external-debt-by-country.php",
    "https://www.globalfirepower.com/purchasing-power-parity.php",
    "https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php",

    "https://www.globalfirepower.com/major-serviceable-airports-by-country.php",
    "https://www.globalfirepower.com/labor-force-by-country.php",
    "https://www.globalfirepower.com/major-ports-and-terminals.php",
    "https://www.globalfirepower.com/merchant-marine-strength-by-country.php",

    "https://www.globalfirepower.com/railway-coverage.php",
    "https://www.globalfirepower.com/roadway-coverage.php",

    "https://www.globalfirepower.com/oil-production-by-country.php",
    "https://www.globalfirepower.com/oil-consumption-by-country.php",
    "https://www.globalfirepower.com/proven-oil-reserves-by-country.php",
    "https://www.globalfirepower.com/natural-gas-production-by-country.php",
    "https://www.globalfirepower.com/natural-gas-consumption-by-country.php",
    "https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php",
    "https://www.globalfirepower.com/coal-production-by-country.php",
    "https://www.globalfirepower.com/coal-consumption-by-country.php",
    "https://www.globalfirepower.com/proven-coal-reserves-by-country.php",

    "https://www.globalfirepower.com/square-land-area.php",
    "https://www.globalfirepower.com/coastline-coverage.php",
    "https://www.globalfirepower.com/border-coverage.php",
    "https://www.globalfirepower.com/waterway-coverage.php"
]

In [ ]:
import pandas as pd

urls_data = pd.DataFrame(urls, columns=["all_url"])
urls_data.index = range(1, len(urls_data) + 1) # it replaces the default index and starts numbering rows from 1 instead of 0

print(f"Total URLs count: {len(urls_data)}")
urls_data.head()

Total URLs count: 54


,all_url
1,https://www.globalfirepower.com/total-populati...
2,https://www.globalfirepower.com/available-mili...
3,https://www.globalfirepower.com/manpower-fit-f...
4,https://www.globalfirepower.com/manpower-reach...
5,https://www.globalfirepower.com/active-militar...


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
df_base = df_power_index.copy()
df_base.set_index("Country_Full_Name", inplace=True) # sets the country name as the index for easier and display the first five rows to verify the changes.
df_base.head()

,Rank,Country_Short_Name,Power_Index
Country_Full_Name,,,
United States,1,USA,0.0744
Russia,2,RUS,0.0788
China,3,CHN,0.0788
India,4,IND,0.1184
South Korea,5,SKO,0.1656


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {"User-Agent": "Mozilla/5.0"}



metric_map = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'total_population',
    'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower',
    'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
    'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually',
    'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
    'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
    'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
    'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
    'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
    'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
    'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
    'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
    'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
    'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
    'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
    'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',
    'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
    'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
    'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
    'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
    'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',
    'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
    'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
    'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
    'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
    'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
    'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
    'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
    'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
    'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',
    'https://www.globalfirepower.com/navy-mine-warfare-craft.php': 'mine_warfare_craft',
    'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
    'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
    'https://www.globalfirepower.com/purchasing-power-parity.php': 'purchasing_power_parity_usd',
    'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'foreign_exchange_and_gold_reserves_usd',
    'https://www.globalfirepower.com/major-serviceable-airports-by-country.php': 'total_serviceable_airports',
    'https://www.globalfirepower.com/labor-force-by-country.php': 'labour_force',
    'https://www.globalfirepower.com/major-ports-and-terminals.php': 'major_ports_and_terminals',
    'https://www.globalfirepower.com/merchant-marine-strength-by-country.php': 'total_merchant_marine_fleet',
    'https://www.globalfirepower.com/railway-coverage.php': 'railway_coverage_km',
    'https://www.globalfirepower.com/roadway-coverage.php': 'roadway_coverage_km',
    'https://www.globalfirepower.com/oil-production-by-country.php': 'oil_production_bbl',
    'https://www.globalfirepower.com/oil-consumption-by-country.php': 'oil_consumption_bbl',
    'https://www.globalfirepower.com/proven-oil-reserves-by-country.php': 'proven_oil_reserves_bbl',
    'https://www.globalfirepower.com/natural-gas-production-by-country.php': 'natural_gas_production_cum',
    'https://www.globalfirepower.com/natural-gas-consumption-by-country.php': 'natural_gas_consumption_cum',
    'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php': 'proven_natural_gas_reserves_cum',
    'https://www.globalfirepower.com/coal-production-by-country.php': 'coal_production_cum',
    'https://www.globalfirepower.com/coal-consumption-by-country.php': 'coal_consumption_mt',
    'https://www.globalfirepower.com/proven-coal-reserves-by-country.php': 'proven_coal_reserves_cum',
    'https://www.globalfirepower.com/square-land-area.php': 'total_land_area_sq_km',
    'https://www.globalfirepower.com/coastline-coverage.php': 'coastline_coverage_km',
    'https://www.globalfirepower.com/border-coverage.php': 'border_coverage_km',
    'https://www.globalfirepower.com/waterway-coverage.php': 'waterway_coverage_km'
}


In [ ]:
def scrape_metric(url, metric_name, df_base):
    print(f"Scraping {metric_name}")

    r = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(r.text, "lxml")

    for card in soup.select("a[href*='country-military-strength-detail']"):
        full = card.select_one(".longFormName span")
        short = card.select_one(".shortFormName span")
        value = card.select_one(".valueContainer span span")

        if full and value:
            country = full.text.strip() # removes extra spaces

            if country not in df_base.index:# if this country is not ready then adds it and it stores the short contry name
                df_base.loc[country, "Country_Short_Name"] = (
                    short.text.strip() if short else None
                )

            df_base.loc[country, metric_name] = value.text.strip()

In [ ]:
for url, metric in metric_map.items():
    scrape_metric(url, metric, df_base)
    time.sleep(2) # makes the program wait for 2 sec before the next request.


Scraping total_population
Scraping total_military_manpower
Scraping fit_for_service
Scraping population_reaching_military_age_annually
Scraping active_personnel
Scraping reserve_personnel
Scraping paramilitary
Scraping total_military_aircraft
Scraping fighter_aircraft
Scraping attack_aircraft
Scraping transport_aircraft
Scraping trainer_aircraft
Scraping special_mission_aircraft
Scraping tanker_aircraft
Scraping total_military_helicopters
Scraping attack_helicopters
Scraping tanks
Scraping armored_fighting_vehicles
Scraping self_propelled_artillery
Scraping towed_artillery
Scraping rocket_projectors
Scraping total_naval_fleet
Scraping total_naval_fleet_tonnage_mt
Scraping aircraft_carriers
Scraping helicopter_carriers
Scraping submarines
Scraping destroyers
Scraping frigates
Scraping corvettes
Scraping coastal_patrol_craft
Scraping mine_warfare_craft
Scraping defense_budget_usd
Scraping external_debt_usd
Scraping purchasing_power_parity_usd
Scraping foreign_exchange_and_gold_reserves_u

In [ ]:
df_final = df_base.reset_index()

df_final.to_csv("/content/drive/MyDrive/Colab Notebooks/project/military_raw_data.csv",
    index=False
)

df_final.head() # displays the first 5 rows of the final datasets.

,Country_Full_Name,Rank,Country_Short_Name,Power_Index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,United States,1,USA,0.0744,"341,963,408","150,463,900","124,816,644","4,445,524","1,328,000","799,500",...,"1,029,000,000,000 \t\t\...","914,301,000,000 \t\t\t\...","13,402,000,000,000 \t\t...","548,849,000 \t\t\t\t\t\...","476,044,000 \t\t\t\t\t\...","248,941,000,000 \t\t\t\...","9,833,517 \t\t\t\t\t\t\...","19,924 \t\t\t\t\t\t\n\t...","12,002 \t\t\t\t\t\t\n\t...","41,009 \t\t\t\t\t\t\n\t..."
1,Russia,2,RUS,0.0788,"140,820,810","69,002,197","46,189,226","1,267,387","1,320,000","2,000,000",...,"617,830,000,000 \t\t\t\...","472,239,000,000 \t\t\t\...","47,805,000,000,000 \t\t...","508,190,000 \t\t\t\t\t\...","310,958,000 \t\t\t\t\t\...","162,166,000,000 \t\t\t\...","17,098,242 \t\t\t\t\t\t...","37,653 \t\t\t\t\t\t\n\t...","22,407 \t\t\t\t\t\t\n\t...","102,000 \t\t\t\t\t\t\n\..."
2,China,3,CHN,0.0788,"1,415,043,270","764,123,366","626,864,169","19,810,606","2,035,000","510,000",...,"225,341,000,000 \t\t\t\...","366,160,000,000 \t\t\t\...","6,654,000,000,000 \t\t\...","4,827,000,000 \t\t\t\t\...","5,313,000,000 \t\t\t\t\...","143,197,000,000 \t\t\t\...","9,596,960 \t\t\t\t\t\t\...","14,500 \t\t\t\t\t\t\n\t...","22,457 \t\t\t\t\t\t\n\t...","27,700 \t\t\t\t\t\t\n\t..."
3,India,4,IND,0.1184,"1,409,128,296","662,290,299","522,786,598","23,955,181","1,455,550","1,155,000",...,"33,170,000,000 \t\t\t\t...","58,867,000,000 \t\t\t\t...","1,381,000,000,000 \t\t\...","985,671,000 \t\t\t\t\t\...","1,200,000,000 \t\t\t\t\...","111,052,000,000 \t\t\t\...","3,287,263 \t\t\t\t\t\t\...","7,000 \t\t\t\t\t\t\n\t\...","13,888 \t\t\t\t\t\t\n\t...","14,500 \t\t\t\t\t\t\n\t..."
4,South Korea,5,SKO,0.1656,"52,081,799","26,040,900","21,353,538","416,654","600,000","3,100,000",...,"55,127,000 \t\t\t\t\t\t...","59,480,000,000 \t\t\t\t...","7,079,000,000 \t\t\t\t\...","15,595,000 \t\t\t\t\t\t...","136,413,000 \t\t\t\t\t\...","326,000,000 \t\t\t\t\t\...","99,720 \t\t\t\t\t\t\n\t...","2,413 \t\t\t\t\t\t\n\t\...",237 \t\t\t\t\t\t\n\t\t\...,"1,600 \t\t\t\t\t\t\n\t\..."


In [ ]:
df_final.head() # to verify that everything looks correct

,Country_Full_Name,Rank,Country_Short_Name,Power_Index,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,United States,1,USA,0.0744,"341,963,408","150,463,900","124,816,644","4,445,524","1,328,000","799,500",...,"1,029,000,000,000 \t\t\...","914,301,000,000 \t\t\t\...","13,402,000,000,000 \t\t...","548,849,000 \t\t\t\t\t\...","476,044,000 \t\t\t\t\t\...","248,941,000,000 \t\t\t\...","9,833,517 \t\t\t\t\t\t\...","19,924 \t\t\t\t\t\t\n\t...","12,002 \t\t\t\t\t\t\n\t...","41,009 \t\t\t\t\t\t\n\t..."
1,Russia,2,RUS,0.0788,"140,820,810","69,002,197","46,189,226","1,267,387","1,320,000","2,000,000",...,"617,830,000,000 \t\t\t\...","472,239,000,000 \t\t\t\...","47,805,000,000,000 \t\t...","508,190,000 \t\t\t\t\t\...","310,958,000 \t\t\t\t\t\...","162,166,000,000 \t\t\t\...","17,098,242 \t\t\t\t\t\t...","37,653 \t\t\t\t\t\t\n\t...","22,407 \t\t\t\t\t\t\n\t...","102,000 \t\t\t\t\t\t\n\..."
2,China,3,CHN,0.0788,"1,415,043,270","764,123,366","626,864,169","19,810,606","2,035,000","510,000",...,"225,341,000,000 \t\t\t\...","366,160,000,000 \t\t\t\...","6,654,000,000,000 \t\t\...","4,827,000,000 \t\t\t\t\...","5,313,000,000 \t\t\t\t\...","143,197,000,000 \t\t\t\...","9,596,960 \t\t\t\t\t\t\...","14,500 \t\t\t\t\t\t\n\t...","22,457 \t\t\t\t\t\t\n\t...","27,700 \t\t\t\t\t\t\n\t..."
3,India,4,IND,0.1184,"1,409,128,296","662,290,299","522,786,598","23,955,181","1,455,550","1,155,000",...,"33,170,000,000 \t\t\t\t...","58,867,000,000 \t\t\t\t...","1,381,000,000,000 \t\t\...","985,671,000 \t\t\t\t\t\...","1,200,000,000 \t\t\t\t\...","111,052,000,000 \t\t\t\...","3,287,263 \t\t\t\t\t\t\...","7,000 \t\t\t\t\t\t\n\t\...","13,888 \t\t\t\t\t\t\n\t...","14,500 \t\t\t\t\t\t\n\t..."
4,South Korea,5,SKO,0.1656,"52,081,799","26,040,900","21,353,538","416,654","600,000","3,100,000",...,"55,127,000 \t\t\t\t\t\t...","59,480,000,000 \t\t\t\t...","7,079,000,000 \t\t\t\t\...","15,595,000 \t\t\t\t\t\t...","136,413,000 \t\t\t\t\t\...","326,000,000 \t\t\t\t\t\...","99,720 \t\t\t\t\t\t\n\t...","2,413 \t\t\t\t\t\t\n\t\...",237 \t\t\t\t\t\t\n\t\t\...,"1,600 \t\t\t\t\t\t\n\t\..."


In [ ]:
df_final.shape # total no rows and columns

(145, 58)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 58 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Country_Full_Name                          145 non-null    object
 1   Rank                                       145 non-null    int64 
 2   Country_Short_Name                         145 non-null    object
 3   Power_Index                                145 non-null    object
 4   total_population                           145 non-null    object
 5   total_military_manpower                    145 non-null    object
 6   fit_for_service                            145 non-null    object
 7   population_reaching_military_age_annually  145 non-null    object
 8   active_personnel                           145 non-null    object
 9   reserve_personnel                          145 non-null    object
 10  paramilitary                          

In [ ]:
df_data_type = df_final.dtypes
df_data_type # it is use to confirm that numeric data is correctly stored and ready to analysis.

,0
Country_Full_Name,object
Rank,int64
Country_Short_Name,object
Power_Index,object
total_population,object
total_military_manpower,object
fit_for_service,object
population_reaching_military_age_annually,object
active_personnel,object
reserve_personnel,object


In [ ]:
df_check_null = df_final.isnull().sum() # checks each column for missing values
df_check_null

,0
Country_Full_Name,0
Rank,0
Country_Short_Name,0
Power_Index,0
total_population,0
total_military_manpower,0
fit_for_service,0
population_reaching_military_age_annually,0
active_personnel,0
reserve_personnel,0


In [ ]:
x = df_final.total_naval_fleet_tonnage_mt
print(x)

0      4,168,037
1      1,260,447
2      2,857,143
3        593,603
4        344,786
         ...    
140          NaN
141          NaN
142          NaN
143          NaN
144          NaN
Name: total_naval_fleet_tonnage_mt, Length: 145, dtype: object


In [ ]:
# Clean Symbols (handles commas, units, multiline values)
import re
import pandas as pd

df_clean = df_final.copy()

def clean_numeric(val):
    if pd.isna(val):
        return None

    val = str(val)

    # 1. Remove line breaks, tabs, extra spaces
    val = re.sub(r"\s+", " ", val)

    # 2. Convert to lowercase
    val = val.lower()

    # 3. Remove commas and common symbols
    val = re.sub(r"[,$%+]", "", val)

    # 4. Remove unit labels (handles Cu.M, mt, etc.)
    val = re.sub(r"(cu\.m|cum|mt|bbl|km|usd)", "", val)

    return val.strip()


In [ ]:
id_columns = ["Country_Full_Name", "Country_Short_Name"]

numeric_columns = [col for col in df_clean.columns if col not in id_columns]

for col in numeric_columns:
    df_clean[col] = df_clean[col].apply(clean_numeric)

In [ ]:
# Power Index → float
df_clean["Power_Index"] = pd.to_numeric(df_clean["Power_Index"], errors="coerce")

# All other numeric columns → int
int_columns = [
    col for col in numeric_columns if col != "Power_Index"
]

df_clean[int_columns] = df_clean[int_columns].apply(
    pd.to_numeric, errors="coerce"
).astype("Int64")


In [ ]:
df_clean[
    [
        "natural_gas_production_cum",
        "natural_gas_consumption_cum",
        "proven_natural_gas_reserves_cum",
        "coal_production_cum",
        "coal_consumption_mt",
        "proven_coal_reserves_cum"
    ]
].isna().sum()

,0
natural_gas_production_cum,0
natural_gas_consumption_cum,0
proven_natural_gas_reserves_cum,0
coal_production_cum,0
coal_consumption_mt,0
proven_coal_reserves_cum,0


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 58 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Country_Full_Name                          145 non-null    object 
 1   Rank                                       145 non-null    Int64  
 2   Country_Short_Name                         145 non-null    object 
 3   Power_Index                                145 non-null    float64
 4   total_population                           145 non-null    Int64  
 5   total_military_manpower                    145 non-null    Int64  
 6   fit_for_service                            145 non-null    Int64  
 7   population_reaching_military_age_annually  145 non-null    Int64  
 8   active_personnel                           145 non-null    Int64  
 9   reserve_personnel                          145 non-null    Int64  
 10  paramilitary              

In [ ]:
df_clean.isnull().sum()

,0
Country_Full_Name,0
Rank,0
Country_Short_Name,0
Power_Index,0
total_population,0
total_military_manpower,0
fit_for_service,0
population_reaching_military_age_annually,0
active_personnel,0
reserve_personnel,0


In [ ]:
df_clean["total_naval_fleet_tonnage_mt"] = (
    df_clean["total_naval_fleet_tonnage_mt"].fillna(0)
)


In [ ]:
df_cleaned = df_clean
df_cleaned.isnull().sum()

,0
Country_Full_Name,0
Rank,0
Country_Short_Name,0
Power_Index,0
total_population,0
total_military_manpower,0
fit_for_service,0
population_reaching_military_age_annually,0
active_personnel,0
reserve_personnel,0


In [ ]:
print("scraped_military_raw_data.csv size",df_final.shape)

scraped_military_raw_data.csv size (145, 58)


In [ ]:
print("military_cleaned.csv size",df_cleaned.shape)

military_cleaned.csv size (145, 58)


In [ ]:
df_cleaned.dtypes

,0
Country_Full_Name,object
Rank,Int64
Country_Short_Name,object
Power_Index,float64
total_population,Int64
total_military_manpower,Int64
fit_for_service,Int64
population_reaching_military_age_annually,Int64
active_personnel,Int64
reserve_personnel,Int64


In [ ]:
military_cleaned = df_cleaned.to_csv("/content/drive/MyDrive/Colab Notebooks/project/military_raw_data.csv",
    index=False
)